In [11]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# WorldCereal dataset
dataset = ee.ImageCollection('ESA/WorldCereal/2021/MODELS/v100')

def mask_other(img):
    return img.updateMask(img.neq(0))

dataset = dataset.map(mask_other)

# Barbados boundary
countries = ee.FeatureCollection('FAO/GAUL/2015/level0')

barbados = countries.filter(
    ee.Filter.eq('ADM0_NAME', 'Barbados')
)

# Convert Barbados feature collection to geometry
barbados_geom = barbados.geometry()

# Temporary crops mosaic
temporarycrops = (
    dataset
    .filter(ee.Filter.eq('product', 'temporarycrops'))
    .mosaic()
    .clip(barbados)
)

# Maize product in Barbados (if available)
maize = (
    dataset
    .filter(ee.Filter.eq('product', 'maize'))
    .mosaic()
    .clip(barbados)
)

# Irrigation product in Barbados (if available)
irrigation = (
    dataset
    .filter(ee.Filter.eq('product', 'irrigation'))
    .mosaic()
    .clip(barbados)
)

# Display
Map = geemap.Map()

Map.centerObject(barbados, 11)

Map.addLayer(
    temporarycrops,
    {
        'bands': ['classification'],
        'min': 0,
        'max': 100,
        'palette': ['ff0000']
    },
    'Temporary Crops'
)

Map.addLayer(
    maize,
    {
        'bands': ['classification'],
        'min': 0,
        'max': 100,
        'palette': ['ebc334']
    },
    'Maize'
)

Map.addLayer(
    irrigation,
    {
        'bands': ['classification'],
        'min': 0,
        'max': 100,
        'palette': ['2d79eb']
    },
    'Irrigation'
)

# Map.addLayer(barbados, {'color': 'white'}, 'Barbados Boundary')

Map

#worldcereal_stack = (
#    temporarycrops.rename('temporarycrops')
#    .addBands(maize.rename('maize'))
#    .addBands(irrigation.rename('irrigation'))
#)

# Define export region
# region = barbados.geometry()

# Create export task
# task = ee.batch.Export.image.toDrive(
#    image=worldcereal_stack,
#    description='worldcereal_barbados',
#    folder='GEE_exports',   # folder in Google Drive
#    fileNamePrefix='worldcereal_barbados',
#    region=region,
#    scale=10,               # adjust if needed
#    crs='EPSG:4326',
#    fileFormat='GeoTIFF',
#    maxPixels=1e13
#)

# task.start()

# print(task.status())

# Temporary Crops
task_temporarycrops = ee.batch.Export.image.toDrive(
    image=temporarycrops,
    description='Barbados_Temporary_Crops',
    folder='GEE_Exports',
    fileNamePrefix='temporary_crops',
    region=barbados_geom,
    scale=10,      # adjust to your dataset resolution
    maxPixels=1e13
)
task_temporarycrops.start()

# Nitrogen
task_maize = ee.batch.Export.image.toDrive(
    image=maize,
    description='Barbados_Maize',
    folder='GEE_Exports',
    fileNamePrefix='maize',
    region=barbados_geom,
    scale=10,
    maxPixels=1e13
)
task_maize.start()

# Sand
task_irrigation = ee.batch.Export.image.toDrive(
    image=irrigation,
    description='Barbados_Irrigation',
    folder='GEE_Exports',
    fileNamePrefix='irrigation',
    region=barbados_geom,
    scale=10,
    maxPixels=1e13
)
task_irrigation.start()